# FloodLens — Phase 4 (Final): Roboflow Import + Local Domain Adaptation
### Upload your Roboflow YOLOv8-segmentation zip, convert to masks, then run domain adaptation

**Before running:** `Runtime > Change runtime type > T4 GPU`. You need a completed
Phase 2 checkpoint already in Drive.

This version fixes the mount/folder issues from earlier attempts: the Drive mount
step checks whether it's already mounted instead of crashing, and every later step
verifies the previous one actually succeeded before continuing, instead of assuming.


## Setup

In [ ]:
!pip install -q segmentation-models-pytorch opencv-python-headless
print("Dependencies installed.")


In [ ]:
import os
import re
import shutil
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import segmentation_models_pytorch as smp

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## 1. Mount Drive (safe against 'already mounted' errors)

In [ ]:
from google.colab import drive

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
else:
    print("Drive already mounted, skipping.")

LOCAL_DATA_DIR = "/content/drive/MyDrive/FloodLens_local_data"
LOCAL_IMAGE_DIR = os.path.join(LOCAL_DATA_DIR, "images")
LOCAL_MASK_DIR = os.path.join(LOCAL_DATA_DIR, "masks")
os.makedirs(LOCAL_IMAGE_DIR, exist_ok=True)
os.makedirs(LOCAL_MASK_DIR, exist_ok=True)

CHECKPOINT_DIR = "/content/drive/MyDrive/FloodLens_checkpoints"
phase2_checkpoint_path = os.path.join(CHECKPOINT_DIR, "best_model.pt")

assert os.path.exists(phase2_checkpoint_path), f"No Phase 2 checkpoint at {phase2_checkpoint_path} — run Phase 2 first."
print("Drive mounted and folders ready.")
print(f"Images will go to: {LOCAL_IMAGE_DIR}")
print(f"Masks will go to: {LOCAL_MASK_DIR}")
print(f"Phase 2 checkpoint confirmed: {phase2_checkpoint_path}")


---
# Part A: Upload and convert your Roboflow export

## 2. Upload the zip
Run this cell, then click "Choose Files" and select your Roboflow YOLOv8
(segmentation) export zip from your computer. If you've already uploaded it to
`/content` in a previous cell, skip this and just set `ZIP_PATH` manually below.


In [ ]:
from google.colab import files

uploaded = files.upload()
ZIP_PATH = "/content/" + list(uploaded.keys())[0]
print(f"Uploaded: {ZIP_PATH}")


## 3. Unzip and inspect the export structure

In [ ]:
EXPORT_DIR = "/content/roboflow_export"

if os.path.isdir(EXPORT_DIR):
    shutil.rmtree(EXPORT_DIR)   # clean slate, avoid mixing with a stale previous export

os.makedirs(EXPORT_DIR, exist_ok=True)
import zipfile
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXPORT_DIR)

print("Unzipped. Top-level contents:")
print(os.listdir(EXPORT_DIR))

for split in ["train", "valid", "test"]:
    split_path = os.path.join(EXPORT_DIR, split)
    if os.path.isdir(split_path):
        print(f"\n{split}/: {os.listdir(split_path)}")


## 4. Verify label format before converting
Checks a real label file's content — a bounding-box export has exactly 5 values
per line, a segmentation (polygon) export has 7+ variable-length values. This
must show polygon-style data or the mask conversion below will silently produce
blank masks.


In [ ]:
def find_first_nonempty_label(export_dir):
    for split in ["train", "valid", "test", ""]:
        labels_dir = os.path.join(export_dir, split, "labels") if split else os.path.join(export_dir, "labels")
        if not os.path.isdir(labels_dir):
            continue
        for fname in sorted(os.listdir(labels_dir)):
            path = os.path.join(labels_dir, fname)
            with open(path) as f:
                content = f.read().strip()
            if content:
                return path, content.split("\n")[0]
    return None, None

label_path, first_line = find_first_nonempty_label(EXPORT_DIR)

if label_path is None:
    print("WARNING: no non-empty label files found anywhere in the export. Check the export itself.")
else:
    num_values = len(first_line.split())
    print(f"Checked: {label_path}")
    print(f"First line: {first_line[:100]}")
    print(f"Number of values: {num_values}")
    if num_values == 5:
        print("\nWARNING: this looks like a BOUNDING BOX export (exactly 5 values), not segmentation.")
        print("Go back to Roboflow and export as YOLOv8 SEGMENTATION specifically, not object detection.")
    elif num_values >= 7:
        print("\nThis looks like a proper polygon/segmentation export. Safe to continue.")
    else:
        print(f"\nUnexpected value count ({num_values}) — inspect manually before continuing.")


## 5. Convert YOLO polygons to PNG masks, save images+masks to Drive
Reads each polygon (normalized 0-1 coordinates), scales to pixel coordinates,
fills the shape to produce a mask. Images with an empty label file (no water
annotated) get an all-background mask, which is noted explicitly rather than
silently produced.


In [ ]:
converted = 0
skipped_no_label = 0
empty_label_images = []

for split in ["train", "valid", "test", ""]:
    images_dir = os.path.join(EXPORT_DIR, split, "images") if split else os.path.join(EXPORT_DIR, "images")
    labels_dir = os.path.join(EXPORT_DIR, split, "labels") if split else os.path.join(EXPORT_DIR, "labels")

    if not os.path.isdir(images_dir):
        continue

    print(f"Processing split: '{split or '(root)'}' -> {images_dir}")

    for img_name in sorted(os.listdir(images_dir)):
        stem, ext = os.path.splitext(img_name)
        label_path = os.path.join(labels_dir, stem + ".txt")

        img_path = os.path.join(images_dir, img_name)
        img = cv2.imread(img_path)
        if img is None:
            print(f"  Could not read image: {img_name}, skipping")
            continue
        h, w = img.shape[:2]

        mask = np.zeros((h, w), dtype=np.uint8)

        if not os.path.exists(label_path):
            skipped_no_label += 1
            print(f"  WARNING: no label file at all for {img_name}")
        else:
            with open(label_path, "r") as f:
                content = f.read().strip()

            if content:
                for line in content.split("\n"):
                    parts = line.strip().split()
                    if len(parts) < 7:
                        continue
                    coords = list(map(float, parts[1:]))
                    points = []
                    for i in range(0, len(coords), 2):
                        x = int(coords[i] * w)
                        y = int(coords[i + 1] * h)
                        points.append([x, y])
                    if len(points) >= 3:
                        cv2.fillPoly(mask, np.array([points], dtype=np.int32), color=255)
            else:
                empty_label_images.append(img_name)

        shutil.copy(img_path, os.path.join(LOCAL_IMAGE_DIR, img_name))
        cv2.imwrite(os.path.join(LOCAL_MASK_DIR, stem + ".png"), mask)
        converted += 1

print(f"\nDone. Converted {converted} image/mask pairs.")
if skipped_no_label:
    print(f"WARNING: {skipped_no_label} images had no label file at all.")
if empty_label_images:
    print(f"NOTE: {len(empty_label_images)} images had an empty label (no water annotated), saved as blank masks:")
    print(f"  {empty_label_images[:5]}")
print(f"Images: {LOCAL_IMAGE_DIR}")
print(f"Masks: {LOCAL_MASK_DIR}")


## 6. Verify the conversion actually landed in Drive
Don't trust the "Done" message alone — confirm the files are really there.


In [ ]:
image_count = len(os.listdir(LOCAL_IMAGE_DIR))
mask_count = len(os.listdir(LOCAL_MASK_DIR))
print(f"Files now in {LOCAL_IMAGE_DIR}: {image_count}")
print(f"Files now in {LOCAL_MASK_DIR}: {mask_count}")

assert image_count > 0, "No images landed in Drive — something went wrong in Section 5, check its output above."
assert mask_count > 0, "No masks landed in Drive — something went wrong in Section 5, check its output above."
print("\nConfirmed non-empty. Safe to continue to Part B.")


---
# Part B: Match, verify, and load data


In [ ]:
def stem(filename):
    return os.path.splitext(filename)[0].lower()

image_files = sorted(os.listdir(LOCAL_IMAGE_DIR))
mask_files = sorted(os.listdir(LOCAL_MASK_DIR))

mask_by_stem = {stem(m): m for m in mask_files}
pairs = []
unmatched = []
for img_f in image_files:
    s = stem(img_f)
    if s in mask_by_stem:
        pairs.append((img_f, mask_by_stem[s]))
    else:
        unmatched.append(img_f)

print(f"Matched {len(pairs)} image/mask pairs")
if unmatched:
    print(f"WARNING: {len(unmatched)} images unmatched. First few: {unmatched[:5]}")

assert len(pairs) > 0, "No pairs matched — check filenames in both folders manually."


## Local dataset class

In [ ]:
IMG_SIZE = 256

class LocalFloodDataset(Dataset):
    def __init__(self, image_dir, mask_dir, pairs, img_size=256):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.pairs = pairs
        self.img_size = img_size

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_name, mask_name = self.pairs[idx]

        img = cv2.imread(os.path.join(self.image_dir, img_name))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (self.img_size, self.img_size))
        img_norm = img.astype(np.float32) / 255.0
        img_t = torch.from_numpy(img_norm).permute(2, 0, 1)

        mask = cv2.imread(os.path.join(self.mask_dir, mask_name), cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, (self.img_size, self.img_size), interpolation=cv2.INTER_NEAREST)
        mask_binary = (mask > 0).astype(np.float32)
        mask_t = torch.from_numpy(mask_binary).unsqueeze(0)

        return img_t, mask_t

local_dataset = LocalFloodDataset(LOCAL_IMAGE_DIR, LOCAL_MASK_DIR, pairs, img_size=IMG_SIZE)
print(f"Local dataset ready with {len(local_dataset)} samples")

sample_img, sample_mask = local_dataset[0]
print("Image shape:", sample_img.shape)
print("Mask shape:", sample_mask.shape)
print("Fraction of water pixels in sample 0:", sample_mask.mean().item())


## Visual sanity check\nActually look at this — with 32 images, one misaligned pair matters a lot.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for i in range(min(4, len(local_dataset))):
    img, mask = local_dataset[i]
    axes[0, i].imshow(img.permute(1, 2, 0).numpy())
    axes[0, i].set_title(f"Local image {i}")
    axes[1, i].imshow(mask[0].numpy(), cmap="Blues")
    axes[1, i].set_title(f"Local mask {i}")
    axes[0, i].axis("off")
    axes[1, i].axis("off")
plt.tight_layout()
plt.show()


---
# Part C: Zero-shot evaluation, fine-tuning, before/after comparison


## Load the Phase 2 model (FloodNet-only, no local fine-tuning yet)

In [ ]:
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
).to(device)

checkpoint = torch.load(phase2_checkpoint_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
print(f"Loaded Phase 2 checkpoint, recorded FloodNet val_iou={checkpoint['val_iou']:.4f}")


## Zero-shot evaluation on local data\n**This is your domain-gap evidence.**

In [ ]:
def compute_iou_batch(logits, targets, threshold=0.5):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    intersection = (preds * targets).sum(dim=(1,2,3))
    union = ((preds + targets) > 0).float().sum(dim=(1,2,3))
    iou = (intersection + 1e-6) / (union + 1e-6)
    return iou.mean().item()

local_loader_full = DataLoader(local_dataset, batch_size=8, shuffle=False, num_workers=2)

model.eval()
zero_shot_ious = []
with torch.no_grad():
    for imgs_b, masks_b in local_loader_full:
        imgs_b, masks_b = imgs_b.to(device), masks_b.to(device)
        logits = model(imgs_b)
        zero_shot_ious.append(compute_iou_batch(logits, masks_b))

zero_shot_iou = np.mean(zero_shot_ious)
print(f"Zero-shot IoU on local data: {zero_shot_iou:.4f}")
print(f"FloodNet val_iou (for comparison): {checkpoint['val_iou']:.4f}")
print(f"Domain gap: {checkpoint['val_iou'] - zero_shot_iou:.4f}")


## Train/val split of local data

In [ ]:
val_fraction = 0.2
val_size = max(1, int(len(local_dataset) * val_fraction))
train_size = len(local_dataset) - val_size

local_train, local_val = random_split(
    local_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)
print(f"Local train size: {len(local_train)}")
print(f"Local val size: {len(local_val)}")

if len(local_dataset) < 50:
    print("\nNOTE: small dataset (<50 images). Treat results as a preliminary pilot.")

BATCH_SIZE = min(8, len(local_train))
local_train_loader = DataLoader(local_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
local_val_loader = DataLoader(local_val, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


## Fine-tune on local data\nLow learning rate (1e-5) since we're adapting, not retraining from scratch.

In [ ]:
class DiceBCELoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        probs = torch.sigmoid(logits)
        smooth = 1e-6
        intersection = (probs * targets).sum(dim=(1,2,3))
        dice_score = (2. * intersection + smooth) / (probs.sum(dim=(1,2,3)) + targets.sum(dim=(1,2,3)) + smooth)
        dice_loss = 1 - dice_score.mean()
        return bce_loss + dice_loss

criterion = DiceBCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

FINETUNE_EPOCHS = 10
PATIENCE = 3
LOCAL_CHECKPOINT_DIR = "/content/drive/MyDrive/FloodLens_local_finetuned"
os.makedirs(LOCAL_CHECKPOINT_DIR, exist_ok=True)

best_val_iou = 0.0
epochs_without_improvement = 0
history = {"train_loss": [], "val_iou": []}

for epoch in range(FINETUNE_EPOCHS):
    model.train()
    train_loss_total = 0.0
    for imgs_b, masks_b in local_train_loader:
        imgs_b, masks_b = imgs_b.to(device), masks_b.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            logits = model(imgs_b)
            loss = criterion(logits, masks_b)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss_total += loss.item() * imgs_b.size(0)

    train_loss = train_loss_total / len(local_train)

    model.eval()
    val_ious = []
    with torch.no_grad():
        for imgs_b, masks_b in local_val_loader:
            imgs_b, masks_b = imgs_b.to(device), masks_b.to(device)
            logits = model(imgs_b)
            val_ious.append(compute_iou_batch(logits, masks_b))
    val_iou = np.mean(val_ious)

    history["train_loss"].append(train_loss)
    history["val_iou"].append(val_iou)
    print(f"Epoch {epoch+1}/{FINETUNE_EPOCHS} | train_loss={train_loss:.4f} | local_val_iou={val_iou:.4f}")

    if val_iou > best_val_iou:
        best_val_iou = val_iou
        epochs_without_improvement = 0
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "val_iou": val_iou,
        }, os.path.join(LOCAL_CHECKPOINT_DIR, "best_model_finetuned.pt"))
        print(f"  -> New best local val_iou={val_iou:.4f}, checkpoint saved.")
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f"No improvement for {PATIENCE} epochs, stopping early.")
            break

print(f"\nFine-tuning complete. Best local val IoU: {best_val_iou:.4f}")


## Before/after comparison — the actual result

In [ ]:
print("=" * 50)
print("DOMAIN ADAPTATION RESULTS")
print("=" * 50)
print(f"FloodNet val IoU (Phase 2):              {checkpoint['val_iou']:.4f}")
print(f"Zero-shot on local data (no adapt):       {zero_shot_iou:.4f}")
print(f"Fine-tuned on local data:                 {best_val_iou:.4f}")
print(f"Domain gap before fine-tuning:             {checkpoint['val_iou'] - zero_shot_iou:.4f}")
print(f"Gap remaining after fine-tuning:           {checkpoint['val_iou'] - best_val_iou:.4f}")
print(f"Improvement from fine-tuning:              {best_val_iou - zero_shot_iou:.4f}")
print("=" * 50)

if best_val_iou <= zero_shot_iou:
    print("\nNOTE: fine-tuning did not improve local IoU. Report this honestly —")
    print("possible causes: too few images, noisy labels, or val split too small.")


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
labels = ["FloodNet\n(Phase 2)", "Zero-shot\non local data", "Fine-tuned\non local data"]
values = [checkpoint['val_iou'], zero_shot_iou, best_val_iou]
colors = ["#4C72B0", "#C44E52", "#55A868"]
ax.bar(labels, values, color=colors)
ax.set_ylabel("IoU")
ax.set_title("Domain adaptation: FloodNet vs local data")
plt.tight_layout()
plt.savefig(os.path.join(LOCAL_CHECKPOINT_DIR, "phase4_domain_gap_chart.png"))
plt.show()


## Visual before/after prediction check

In [ ]:
imgs_b, masks_b = next(iter(local_val_loader))
imgs_b = imgs_b.to(device)

model.eval()
with torch.no_grad():
    logits = model(imgs_b)
    preds = (torch.sigmoid(logits) > 0.5).float().cpu()

fig, axes = plt.subplots(3, min(4, imgs_b.shape[0]), figsize=(16, 10))
for i in range(min(4, imgs_b.shape[0])):
    axes[0, i].imshow(imgs_b[i].cpu().permute(1, 2, 0).numpy())
    axes[0, i].set_title(f"Local input {i}")
    axes[1, i].imshow(masks_b[i, 0].numpy(), cmap="Blues")
    axes[1, i].set_title(f"Ground truth {i}")
    axes[2, i].imshow(preds[i, 0].numpy(), cmap="Blues")
    axes[2, i].set_title(f"Fine-tuned prediction {i}")
    for row in range(3):
        axes[row, i].axis("off")
plt.tight_layout()
plt.show()


## Phase 4 complete
- Roboflow export uploaded, verified as segmentation format, converted to masks
- Images and masks confirmed present in Drive before proceeding
- Quantified domain gap between FloodNet and your local data
- Fine-tuned checkpoint saved to Drive

**Next:** Phase 5 — WebXR/AR visualization layer.
